# Breadth-First Search and Paths


```{contents}
:local:
:depth: 2
```


```{index} Discover a Vertex Once, Then Process Its Neighbors
```

## Discover a Vertex Once, Then Process Its Neighbors



A delivery graph has edges A-B, A-C, B-D and C-D. Starting at A, how many roads are needed to reach each stop? Breadth-first search answers a minimum-hop question in an unweighted graph. It keeps a queue of discovered vertices whose outgoing neighbors still need examination. Discovery and processing are different events: a vertex can already be known while it is still waiting in the queue.

Mark a vertex discovered when it is enqueued. If you wait until dequeue, both B and C can enqueue D before D is processed. That duplicates work and makes predecessor choices harder to reason about. A visited/discovered set is therefore part of the algorithm's state, not merely an optional speed optimization.

| Processing event | Queue afterward | Newly discovered distances |
| --- | --- | --- |
| Initialize A | A | A: 0 |
| Examine A | B, C | B: 1, C: 1 |
| Examine B | C, D | D: 2 |
| Examine C | D | None; D is already known |
| Examine D | Empty | None |

The table assumes that A lists B before C. A different neighbor order can choose another equally short predecessor route, but it does not change the minimum hop distance. Tests that insist on one exact path without specifying neighbor order can reject correct results. State the tie policy separately from the distance claim.

For related explanation, see the further reading below.[^development-reading]


```{index} Discover a Vertex Once, Then Process Its Neighbors
```



```{index} Turn Queue Events into a Traversal Loop
```

### Turn Queue Events into a Traversal Loop


A facilities graph answers which rooms can be reached and how many doorways a shortest route uses. Every edge in this lesson contributes one hop. The queue is not a list of completed rooms; it holds discovered rooms whose neighbors still need examination. This is the queue policy from Chapter 15 applied to the graph model from the preceding sections.

Use directed edges A→B, A→C, B→D and C→D, plus an isolated E. Neighbor order is written explicitly. Begin with A discovered, distance zero and no predecessor. The small program exposes the traversal core; the full graph class and validation examples remain below.


In [1]:
using System;
using System.Collections.Generic;
var next = new Dictionary<string, string[]>
{
    ["A"] = new[] { "B", "C" }, ["B"] = new[] { "D" },
    ["C"] = new[] { "D" }, ["D"] = Array.Empty<string>(), ["E"] = Array.Empty<string>()
};
var distance = new Dictionary<string, int> { ["A"] = 0 };
var previous = new Dictionary<string, string>();
var waiting = new Queue<string>();
waiting.Enqueue("A");
while (waiting.Count > 0)
{
    string current = waiting.Dequeue();
    foreach (string neighbor in next[current])
    {
        if (distance.ContainsKey(neighbor)) continue;
        distance[neighbor] = distance[current] + 1;
        previous[neighbor] = current;
        waiting.Enqueue(neighbor);
        Console.WriteLine($"Discover {neighbor}: distance {distance[neighbor]}, from {current}");
    }
}
Console.WriteLine($"D hops: {distance["D"]}; E reached: {distance.ContainsKey("E")}");


Discover B: distance 1, from A
Discover C: distance 1, from A
Discover D: distance 2, from B
D hops: 2; E reached: False


Here membership in `distance` also means discovered, so a separate visited set would duplicate that fact. All endpoints are declared in this teaching graph. Production validation of unknown labels is retained in the complete implementation.

| Statement group | Meaning in the trace | Why the order matters |
| --- | --- | --- |
| Dequeue current | Choose the oldest waiting room | Complete this layer before later layers |
| Test distance membership | Skip a previously discovered neighbor | C must not enqueue D a second time |
| Store distance and predecessor | Record the first discovery | A known room has one chosen route witness |
| Enqueue neighbor | Schedule its remaining work | Mark discovery before another parent can find it |

After A, the queue is B,C. After B, it is C,D; D already has distance two and predecessor B. When C examines D, the membership test succeeds and no new queue entry is added. A cycle back to A is harmless for the same reason: A was marked before processing began.

A predecessor map points backwards. For D, following D←B←A gives a reverse route. Reverse those labels to display A→B→D. A missing destination has no such chain; do not create a one-room route and call it reachable. The source itself is a valid one-room route with zero edges.


**Common mistake.** Marking a vertex when it is dequeued instead of when it is enqueued. In the diamond, both B and C then enqueue D, D is processed twice, and the second processing can overwrite the predecessor recorded by the first.


```{index} Breadth-First Search and Discovery
```

### Breadth-First Search and Discovery




```{index} Breadth-First Search and Discovery
```


Breadth-first search (BFS) discovers vertices in layers. Mark a vertex when it enters the queue, so two incoming edges cannot enqueue it twice. The source has distance zero. Each newly discovered neighbor receives its predecessor distance plus one. Queue order ensures that all vertices at distance d are processed before distance d+1. These are minimum edge counts for an unweighted graph; they are not minimum travel times when weights differ.


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
SearchResult result = graph.Bfs("Lobby");
foreach (string room in graph.Vertices())
    Console.WriteLine($"{room}: {result.DistanceTo(room)?.ToString() ?? "unreachable"}");

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Lobby: 0
Hall: 1
Lab: 2
Office: 2
Remote: unreachable


The output is `Lobby: 0`, `Hall: 1`, `Lab: 2`, `Office: 2`, and `Remote: unreachable`. Lab is two hops away through Hall, even though it is also reachable through Office; BFS recorded the first, shorter discovery. `Remote` prints as unreachable because its distance is `null`, not zero.


**Visualize it.** OpenDSA's [Breadth-First Search](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/GraphTraversal.html#breadth-first-search) slideshow colors vertices as they enter and leave the queue, layer by layer.

```{index} Code Tracing: Queue Contents Layer by Layer
```

### Code Tracing: Queue Contents Layer by Layer

Run BFS by hand from S on the undirected graph with edges S-A, S-B, A-C, B-C, B-D, C-E, D-E, where each vertex lists its neighbors in alphabetical order. After each dequeue write the vertex processed, the queue contents afterward, and any newly assigned distances. Then predict the distance of E and its predecessor. The solution prints the same trace.

In [ ]:
// Solution
using System;
using System.Collections.Generic;

var neighbors = new Dictionary<string, string[]>
{
    ["S"] = new[] { "A", "B" }, ["A"] = new[] { "C", "S" }, ["B"] = new[] { "C", "D", "S" },
    ["C"] = new[] { "A", "B", "E" }, ["D"] = new[] { "B", "E" }, ["E"] = new[] { "C", "D" },
};
var distance = new Dictionary<string, int> { ["S"] = 0 };
var previous = new Dictionary<string, string>();
var queue = new Queue<string>();
queue.Enqueue("S");
while (queue.Count > 0)
{
    string current = queue.Dequeue();
    var discovered = new List<string>();
    foreach (string next in neighbors[current])
    {
        if (distance.ContainsKey(next)) continue;
        distance[next] = distance[current] + 1;
        previous[next] = current;
        queue.Enqueue(next);
        discovered.Add($"{next}={distance[next]}");
    }
    Console.WriteLine($"process {current}: queue [{string.Join(", ", queue)}]  new distances {(discovered.Count == 0 ? "none" : string.Join(" ", discovered))}");
}
Console.WriteLine($"E: distance {distance["E"]}, predecessor {previous["E"]}");

process S: queue [A, B]  new distances A=1 B=1
process A: queue [B, C]  new distances C=2
process B: queue [C, D]  new distances D=2
process C: queue [D, E]  new distances E=3
process D: queue [E]  new distances none
process E: queue []  new distances none
E: distance 3, predecessor C


S discovers A and B at distance 1; A discovers C at distance 2; B discovers D at distance 2 and finds C already known; C discovers E at distance 3 and E's predecessor is C, because C was processed before D. Processing D and E discovers nothing new. The queue never holds two copies of a vertex, and every vertex at distance 2 left the queue before E, at distance 3, entered it, which is the layer property. With D listed before C in B's neighbors, nothing would change here, because C was discovered from A first; the tie that could change E's predecessor is between C and D, and it is settled by which of the two is dequeued first.

```{index} Order Tasks with a Topological Sort
```

### Order Tasks with a Topological Sort

A queue solves a second classic graph problem. Given a DAG of tasks and prerequisite arrows, a **topological sort** lists the tasks so that every arrow points from an earlier task to a later one. Course prerequisites, build steps, approval workflows, and spreadsheet recalculation all need such an order, and a DAG always has at least one; a cycle has none.

The queue-based method works from in-degrees. A task with in-degree zero has no unfinished prerequisite, so it can go first; completing it removes its outgoing arrows, which may drop other tasks' in-degrees to zero, and those join the queue. The loop keeps going until the queue is empty. If some task was never output, its in-degree never reached zero, which means it lies on a cycle. That makes the method a cycle detector as well as a scheduler.

| Step | Queue of ready tasks | Output so far | In-degree changes |
| --- | --- | --- | --- |
| Start | Request, Budget | | Approve 2, Archive 1 |
| Output Request | Budget | Request | Approve 1 |
| Output Budget | Approve | Request, Budget | Approve 0 |
| Output Approve | Archive | Request, Budget, Approve | Archive 0 |
| Output Archive | empty | Request, Budget, Approve, Archive | |

The order is not unique: Budget could have been output before Request. A program that must produce the same order every run adds a tie rule, such as alphabetical order among ready tasks, which the example does by using a sorted set instead of a plain queue.

In [ ]:
using System;
using System.Collections.Generic;

Console.WriteLine("workflow: " + Order(new[] { "Request", "Budget", "Approve", "Archive" },
    new[] { ("Request", "Approve"), ("Budget", "Approve"), ("Approve", "Archive") }));
Console.WriteLine("with a cycle: " + Order(new[] { "Draft", "Review", "Sign" },
    new[] { ("Draft", "Review"), ("Review", "Sign"), ("Sign", "Review") }));

static string Order(string[] tasks, (string From, string To)[] arrows)
{
    var indegree = new Dictionary<string, int>();
    var outgoing = new Dictionary<string, List<string>>();
    foreach (string t in tasks) { indegree[t] = 0; outgoing[t] = new List<string>(); }
    foreach (var (from, to) in arrows) { outgoing[from].Add(to); indegree[to]++; }

    var ready = new SortedSet<string>(StringComparer.Ordinal);      // deterministic tie rule
    foreach (string t in tasks) if (indegree[t] == 0) ready.Add(t);
    var order = new List<string>();
    while (ready.Count > 0)
    {
        string task = ready.Min!;
        ready.Remove(task);
        order.Add(task);
        foreach (string next in outgoing[task])
            if (--indegree[next] == 0) ready.Add(next);               // its last prerequisite is done
    }
    return order.Count == tasks.Length
        ? string.Join(" -> ", order)
        : $"no valid order; {tasks.Length - order.Count} task(s) are on a cycle";
}

workflow: Budget -> Request -> Approve -> Archive
with a cycle: no valid order; 2 task(s) are on a cycle


The workflow comes out as Budget, Request, Approve, Archive: both starting tasks were ready at once and the alphabetical tie rule chose Budget. The second graph outputs Draft and then stops, because Review and Sign each wait for the other; the method reports that two tasks are on a cycle rather than inventing an order. Every arrow is examined once and every task enters the ready set once, so the cost is O(V + E) plus the logarithmic cost of the sorted set. A depth-first version exists too: run DFS and record each vertex when its call *exits*; the reverse of that exit order is a topological order, which 20.4 revisits once exit events have been traced.

```{index} Code Completion: Release the Dependents
```

### Code Completion: Release the Dependents

The topological sort below is missing the step that runs when a task is output: decrease the in-degree of each task it points to, and enqueue any task whose in-degree reaches zero. Complete it so that the output is `Compile -> Test -> Package -> Deploy`.

In [ ]:
using System;
using System.Collections.Generic;

string[] tasks = { "Compile", "Test", "Package", "Deploy" };
(string From, string To)[] arrows = { ("Compile", "Test"), ("Test", "Package"), ("Compile", "Package"), ("Package", "Deploy") };

var indegree = new Dictionary<string, int>();
var outgoing = new Dictionary<string, List<string>>();
foreach (string t in tasks) { indegree[t] = 0; outgoing[t] = new List<string>(); }
foreach (var (from, to) in arrows) { outgoing[from].Add(to); indegree[to]++; }

var ready = new Queue<string>();
foreach (string t in tasks) if (indegree[t] == 0) ready.Enqueue(t);
var order = new List<string>();
while (ready.Count > 0)
{
    string task = ready.Dequeue();
    order.Add(task);
    // Your code starts here.

    // Your code ends here.
}
Console.WriteLine(order.Count == tasks.Length ? string.Join(" -> ", order) : "cycle detected");

In [ ]:
// Solution
using System;
using System.Collections.Generic;

string[] tasks = { "Compile", "Test", "Package", "Deploy" };
(string From, string To)[] arrows = { ("Compile", "Test"), ("Test", "Package"), ("Compile", "Package"), ("Package", "Deploy") };

var indegree = new Dictionary<string, int>();
var outgoing = new Dictionary<string, List<string>>();
foreach (string t in tasks) { indegree[t] = 0; outgoing[t] = new List<string>(); }
foreach (var (from, to) in arrows) { outgoing[from].Add(to); indegree[to]++; }

var ready = new Queue<string>();
foreach (string t in tasks) if (indegree[t] == 0) ready.Enqueue(t);
var order = new List<string>();
while (ready.Count > 0)
{
    string task = ready.Dequeue();
    order.Add(task);
    foreach (string next in outgoing[task])
    {
        indegree[next]--;                          // one prerequisite of next is now done
        if (indegree[next] == 0) ready.Enqueue(next);   // all of them are done
    }
}
Console.WriteLine(order.Count == tasks.Length ? string.Join(" -> ", order) : "cycle detected");

Compile -> Test -> Package -> Deploy


Leaving the block empty outputs only `Compile` and then reports a cycle, because no other task's in-degree ever reaches zero; the method cannot tell a missing update from a real cycle, which is why the completion matters. Package has two prerequisites, so it is enqueued only when the second decrement arrives, after Test; enqueueing on every decrement would output Package twice and Deploy early.

```{index} Recover a Path from Earlier Layers
```



```{index} Recover a Path from Earlier Layers
```

## Recover a Path from Earlier Layers



When discovering a new vertex v from u, record predecessor[v] = u and distance[v] = distance[u] + 1. Following those predecessor links backward reconstructs a route to the source. Reverse the route to display it from source to destination. The source is a one-vertex route to itself; an unreachable known vertex has no source route. Do not fabricate a destination-only path when no predecessor chain exists.

Why is the distance minimal? The queue processes vertices in nondecreasing discovered distance. All one-hop vertices are discovered from the source before processing can expose a two-hop successor, and the same reasoning repeats layer by layer. The first discovery therefore uses a shortest hop count. This reasoning relies on equal edge contribution, not on geometric closeness or vertex spelling. With arbitrary weights, fewer edges can be more expensive; Chapter 24 treats that different objective.

For adjacency lists, visiting reachable vertices and their incident/outgoing entries costs work proportional to those entries plus vertex bookkeeping. A traversal of one component does not visit all disconnected components by magic. If the task is a full undirected component report, start a new traversal from each still-undiscovered vertex, including isolates.

Guided exercise: add an isolated E and a cycle D-A to the graph. Predict which vertices are discovered from A, whether any is enqueued twice, and the result for a path to E. Then choose C before B in A's neighbor list. Explain which outputs may change and which correctness obligations remain fixed.

```{dropdown} Verification checkpoint
E is unreachable from A; its route must remain absent. The cycle does not rediscover A because A was marked at initialization. Either A-B-D or A-C-D can be a shortest route to D under the chosen neighbor order; both use two edges.
```


```{index} Graph Traversal: Reachability, Paths, and Components
```

<span id="graph-traversal-reachability-paths-and-components"></span>


A facilities team needs to know which rooms can be reached, how many unweighted links a route needs, and which rooms form disconnected groups. A representation stores connections; traversal systematically discovers what those connections imply. Start with declared vertices so isolated rooms remain visible. Every example includes the complete model. The implementation uses ordinary classes, loops, a queue, and a visited set.


```{index} Recover a Path with Predecessors
```

### Recover a Path with Predecessors




```{index} Recover a Path with Predecessors
```


Store a predecessor only at first discovery. To recover a route, follow predecessors from the destination back to the source and reverse the collected list. The source route contains the source itself; an unreachable target produces an empty array. Neighbor insertion order chooses among equally short routes. Test the distance and valid edges instead of assuming an arbitrary tie order is a mathematical requirement.


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
SearchResult result = graph.Bfs("Lobby");
Console.WriteLine(string.Join(" -> ", result.PathTo("Lab")));
Console.WriteLine($"Remote route length: {result.PathTo("Remote").Length}");
Console.WriteLine(string.Join(" -> ", result.PathTo("Lobby")));

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Lobby -> Hall -> Lab
Remote route length: 0
Lobby


The output is `Lobby -> Hall -> Lab`, `Remote route length: 0`, and `Lobby`. The route was rebuilt by following predecessors backward from Lab and reversing; the unreachable room gives an empty array rather than a made-up path; and the source's route to itself is the one-vertex list.

**Common mistake.** Returning the predecessor chain without reversing it. The list is collected from the destination backward, so the printed route reads Lab, Hall, Lobby; reverse once before displaying, as `PathTo` does.


```{index} Guide the Route and Tie-Policy Checks
```

### Guide the Route and Tie-Policy Checks


Fill this reverse chain before writing code:

| Current room | Recorded predecessor | Reverse route so far |
| --- | --- | --- |
| D | B | D |
| B | A | D,B |
| A | None; this is the source | D,B,A |

Guided task: change A's neighbors to C,B. Predict the new predecessor of D and the hop count. Then add D→A and check how many times A and D are discovered. Finally, ask for E and specify the result without indexing a missing map entry.

```{dropdown} Check the graph reasoning
D's predecessor becomes C, giving A→C→D, but its distance stays two. The cycle cannot rediscover A because A is already in the distance map. A and D are each discovered once. E is a known but unreachable vertex and has no route from A. Tie-dependent routes can differ while their minimum hop counts agree.
```

For independent application, use the chapter's network tasks to report both distances and route witnesses. Check each route edge and its edge count, rather than trusting one printed distance. The complete BFS and predecessor-path programs in this section provide the graph container, boundary checks and additional practice.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. BFS from the source reports every vertex's distance. Why can the first discovery of a vertex be trusted as its shortest distance?

```{dropdown} Answer
Because the queue processes vertices in nondecreasing distance: every vertex at distance d is processed, and therefore every vertex at distance d + 1 discovered, before any vertex at distance d + 1 is processed. A later discovery of the same vertex would come from a vertex at the same or greater distance and cannot be shorter.
```

2. What changes in BFS output when a vertex lists its neighbors in a different order, and what does not?

```{dropdown} Answer
The predecessor chosen among equally short routes and the visit order can change; the distances cannot. A test that asserts one exact path without fixing the neighbor order can fail on a correct implementation.
```

3. A topological sort of a project's tasks outputs only 17 of 20 tasks. What does that tell the project manager?

```{dropdown} Answer
The remaining three tasks form or depend on a cycle of prerequisites, so no schedule can satisfy them all. The algorithm has found a planning error, and the fix is in the prerequisites, not in the sort.
```

4. Why is BFS the right tool for the fewest number of flights but not for the shortest total flying time?

```{dropdown} Answer
BFS counts edges, so it minimizes hops only when every edge costs the same. Flying times differ per edge, which needs the weighted shortest-path algorithm of Chapter 24; BFS applied to that graph would happily prefer one 12-hour flight over two 2-hour flights.
```

```{rubric} Footnotes
```


[^development-reading]: Further reading: [OpenDSA, Graph Traversals](https://opendsa.org/OpenDSA/Books/Catalog/html/GraphTraversal.html). The worked business scenarios and guided reasoning here are original teaching examples.
